# 📘 Notebook 05 · Solidity + 智能合约

> Notebook 04 我们手写了区块链。现在我们学**在别人的区块链上写程序**——这就是智能合约。

**学完你会：**

- 看懂、写出标准的 Solidity 合约
- 自己写一个 ERC20 代币（你自己的"币"）和一个 NFT
- 用 web3.py 部署合约、调用函数、读链上数据
- 避开 6 个最常见的安全漏洞

**预计时间：8-12 小时**

**前置**：Notebook 04；会一门面向对象语言（Python / Java / JavaScript 都行）。

## 1. Solidity 是什么

Solidity 是**给 EVM 写程序的语言**。

- 语法像 JavaScript + 一点 Python + 一点 C++
- 静态类型、编译到 EVM 字节码
- 主要用途：写智能合约
- 替代品：Vyper（语法像 Python，更安全但生态小）

> EVM 上跑的所有 DeFi 协议（Uniswap、Aave、Compound）、NFT 协议（OpenSea 合约、各大 NFT 项目）几乎都是 Solidity 写的。

### 1.1 Hello World 合约

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract HelloWorld {
    string public message;          // 自动生成 getter 函数

    constructor(string memory _msg) {
        message = _msg;
    }

    function setMessage(string memory _msg) public {
        message = _msg;
    }
}
```

短短几行就引出了 Solidity 的关键概念：

| 关键字 | 含义 |
|---|---|
| `pragma solidity` | 编译器版本约束 |
| `contract` | 类似 class |
| `public` | 自动生成 getter |
| `constructor` | 部署时执行一次 |
| `string memory` | string 类型 + 内存位置 |
| `function` | 函数定义 |

## 2. Solidity 核心语法速览

下面是你需要知道的所有基础语法。不需要背，**先看一遍混个眼熟，写代码时回来查**。

### 2.1 数据类型

```solidity
// 整数
uint256 a = 100;          // 无符号 256 位
int256 b = -50;           // 有符号
uint8 c = 255;            // 0~255

// 地址
address alice = 0x1234...;
address payable bob = payable(alice);    // 可以接收 ETH

// 布尔
bool flag = true;

// 字节 / 字符串
bytes32 hash = keccak256("hello");
string memory name = "Alice";

// 数组
uint[] memory arr = new uint[](5);       // 动态数组（堆）
uint[5] memory fixed_arr;                // 定长数组

// 映射（最常用！）
mapping(address => uint) public balances;          // 类似 dict
mapping(address => mapping(address => uint)) allowances;  // 嵌套
```

### 2.2 存储位置

这是 Solidity **最容易出错**的概念之一：

| 位置 | 含义 | Gas |
|---|---|---|
| `storage` | 链上永久存储 | 贵（20000 gas/SSTORE）|
| `memory` | 函数调用内的临时变量 | 便宜 |
| `calldata` | 外部调用传入的参数（只读）| 最便宜 |

**经验法则**：
- 合约状态变量自动是 storage
- 函数参数 / 局部变量要显式声明 memory 或 calldata
- string、bytes、array、struct 必须声明位置；uint、bool、address 不用

### 2.3 可见性

```solidity
function f() public {}      // 外部和内部都能调
function g() external {}    // 只能从合约外部调（自己合约其他函数调不了）
function h() internal {}    // 自己合约和子合约能调
function i() private {}     // 只有自己合约能调
```

**变量也有这些修饰**。`public` 变量自动生成 getter，所以"公开的变量"= 用了 `public` 关键字。

### 2.4 状态修饰

```solidity
function pure_fn(uint a, uint b) public pure returns(uint) {
    return a + b;     // pure: 不读写链上状态
}

function view_fn() public view returns(uint) {
    return block.number;     // view: 只读不写
}

function write_fn() public {
    counter += 1;            // 默认可读写
}

function payable_fn() public payable {
    // 能接收 ETH 转账
}
```

## 3. 第一个有用合约：ERC20 代币

**ERC20** 是以太坊上代币的**事实标准**——所有 DEX、钱包、协议都按这个接口和代币交互。

只要你的合约实现了 ERC20 的 6 个函数 + 2 个事件，它就是一个"代币"。

### 3.1 ERC20 必须实现的接口

```solidity
function totalSupply() external view returns (uint256);
function balanceOf(address account) external view returns (uint256);
function transfer(address to, uint256 amount) external returns (bool);
function allowance(address owner, address spender) external view returns (uint256);
function approve(address spender, uint256 amount) external returns (bool);
function transferFrom(address from, address to, uint256 amount) external returns (bool);

event Transfer(address indexed from, address indexed to, uint256 value);
event Approval(address indexed owner, address indexed spender, uint256 value);
```

### 3.2 从零写一个

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract MyToken {
    string  public name     = "MyToken";
    string  public symbol   = "MTK";
    uint8   public decimals = 18;
    uint256 public totalSupply;

    mapping(address => uint256) public balanceOf;
    mapping(address => mapping(address => uint256)) public allowance;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10**uint256(decimals);
        balanceOf[msg.sender] = totalSupply;     // 部署者拿到全部
        emit Transfer(address(0), msg.sender, totalSupply);   // 铸造事件
    }

    function transfer(address _to, uint256 _value) public returns (bool) {
        require(balanceOf[msg.sender] >= _value, "balance too low");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }

    function approve(address _spender, uint256 _value) public returns (bool) {
        allowance[msg.sender][_spender] = _value;
        emit Approval(msg.sender, _spender, _value);
        return true;
    }

    function transferFrom(address _from, address _to, uint256 _value)
        public returns (bool)
    {
        require(balanceOf[_from] >= _value, "balance too low");
        require(allowance[_from][msg.sender] >= _value, "allowance too low");
        balanceOf[_from] -= _value;
        balanceOf[_to]   += _value;
        allowance[_from][msg.sender] -= _value;
        emit Transfer(_from, _to, _value);
        return true;
    }
}
```

**这就是一个 ERC20 代币的全部代码。**Uniswap 上能交易的代币，本质都是这 30 行。

## 4. 用 Python 部署、调用合约

我们用 `eth-tester`（本地链）+ `web3.py` 完整跑一遍。**不需要花真钱、不需要装节点**。

In [ ]:
# 一次性安装：
# pip install web3 py-solc-x eth-tester[py-evm]
#
# 第一次用 py-solc-x 要装编译器：
# python -c "import solcx; solcx.install_solc('0.8.20')"

try:
    from web3 import Web3
    from eth_tester import EthereumTester
    from web3.providers.eth_tester import EthereumTesterProvider
    import solcx
    HAS_WEB3 = True
except ImportError as e:
    HAS_WEB3 = False
    print('未安装相关库:', e)
    print('运行: pip install web3 py-solc-x eth-tester[py-evm]')

if HAS_WEB3:
    # 安装 solc 0.8.20（首次要 30s 左右下载）
    try:
        solcx.set_solc_version('0.8.20')
    except Exception:
        print('正在安装 solc 0.8.20…')
        solcx.install_solc('0.8.20')
        solcx.set_solc_version('0.8.20')

    # 连接本地测试链
    w3 = Web3(EthereumTesterProvider(EthereumTester()))
    print('已连接？', w3.is_connected())
    print('当前块高:', w3.eth.block_number)
    print('账户数:', len(w3.eth.accounts))
    print('账户 0 余额:', w3.from_wei(w3.eth.get_balance(w3.eth.accounts[0]), 'ether'), 'ETH')

In [ ]:
if HAS_WEB3:
    # ---------- 1. 写合约源码 ----------
    src = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract MyToken {
    string  public name     = "MyToken";
    string  public symbol   = "MTK";
    uint8   public decimals = 18;
    uint256 public totalSupply;

    mapping(address => uint256) public balanceOf;
    mapping(address => mapping(address => uint256)) public allowance;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10**uint256(decimals);
        balanceOf[msg.sender] = totalSupply;
        emit Transfer(address(0), msg.sender, totalSupply);
    }

    function transfer(address _to, uint256 _value) public returns (bool) {
        require(balanceOf[msg.sender] >= _value, "balance too low");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }

    function approve(address _spender, uint256 _value) public returns (bool) {
        allowance[msg.sender][_spender] = _value;
        emit Approval(msg.sender, _spender, _value);
        return true;
    }

    function transferFrom(address _from, address _to, uint256 _value) public returns (bool) {
        require(balanceOf[_from] >= _value, "balance too low");
        require(allowance[_from][msg.sender] >= _value, "allowance too low");
        balanceOf[_from] -= _value;
        balanceOf[_to]   += _value;
        allowance[_from][msg.sender] -= _value;
        emit Transfer(_from, _to, _value);
        return true;
    }
}
'''

    # ---------- 2. 编译 ----------
    compiled = solcx.compile_source(src, output_values=['abi', 'bin'])
    contract_id, contract_interface = compiled.popitem()
    abi  = contract_interface['abi']
    bytecode = contract_interface['bin']
    print(f'编译完毕：ABI 中函数数量 = {sum(1 for x in abi if x.get("type")=="function")}')
    print(f'字节码长度: {len(bytecode)} 字符')

In [ ]:
if HAS_WEB3:
    # ---------- 3. 部署 ----------
    deployer = w3.eth.accounts[0]
    Contract = w3.eth.contract(abi=abi, bytecode=bytecode)

    tx_hash = Contract.constructor(1_000_000).transact({'from': deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    token_address = receipt.contractAddress
    print('合约部署地址:', token_address)
    print('部署消耗 gas:', receipt.gasUsed)

    # ---------- 4. 与合约交互 ----------
    token = w3.eth.contract(address=token_address, abi=abi)

    print(f'\n名称: {token.functions.name().call()}')
    print(f'符号: {token.functions.symbol().call()}')
    print(f'精度: {token.functions.decimals().call()}')
    print(f'总供应: {token.functions.totalSupply().call() / 10**18}')
    print(f'部署者余额: {token.functions.balanceOf(deployer).call() / 10**18}')

    # ---------- 5. 转账给另一个账户 ----------
    alice = w3.eth.accounts[1]
    tx = token.functions.transfer(alice, 1000 * 10**18).transact({'from': deployer})
    w3.eth.wait_for_transaction_receipt(tx)

    print(f'\n转账后：')
    print(f'部署者余额: {token.functions.balanceOf(deployer).call() / 10**18}')
    print(f'Alice  余额: {token.functions.balanceOf(alice).call() / 10**18}')

    # ---------- 6. 读事件日志 ----------
    transfers = token.events.Transfer().get_logs(from_block=0)
    print(f'\n共 {len(transfers)} 条 Transfer 事件:')
    for ev in transfers:
        a = ev.args
        print(f'  from={a["from"][:10]}... to={a["to"][:10]}... value={a.value/10**18}')

**🎉 你刚刚做完了一个完整的"币圈"流程：**

1. 写合约 ✓
2. 编译 ✓
3. 部署到链 ✓
4. 与合约交互 ✓
5. 读链上事件 ✓

把上面 30 行 Solidity 改个名字 + 改个总量，发到以太坊主网，就是一个"项目方发币"。

历史上无数 $100M+ 市值的项目，合约部分就是这 30 行（其余靠营销和叙事）。

### 🎯 练习

三个练习：

1. 给上面的 ERC20 加一个 `mint(address to, uint256 amount)` 函数，只允许 `owner` 调用
2. 加一个 `burn(uint256 amount)`，销毁调用者持有的代币
3. 测试 Alice 用 `approve` 授权给 Bob 后，Bob 调用 `transferFrom` 转走 Alice 的钱

In [ ]:
# 在这里写你的答案


## 5. OpenZeppelin：不要自己写关键合约

**生产代码不会从零写 ERC20。** 自己写容易留漏洞，标准做法是用 [OpenZeppelin](https://github.com/OpenZeppelin/openzeppelin-contracts) 经审计的库。

### 5.1 生产环境的 ERC20 长这样

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

import "@openzeppelin/contracts/token/ERC20/ERC20.sol";
import "@openzeppelin/contracts/access/Ownable.sol";

contract MyToken is ERC20, Ownable {
    constructor() ERC20("MyToken", "MTK") Ownable(msg.sender) {
        _mint(msg.sender, 1_000_000 * 10**decimals());
    }

    function mint(address to, uint256 amount) external onlyOwner {
        _mint(to, amount);
    }
}
```

只有 10 行你自己写的代码，剩下都靠 `ERC20` 基类。**已被审计过几百次，比你自己写靠谱得多。**

### 5.2 OpenZeppelin 主要模块

| 模块 | 用途 |
|---|---|
| `ERC20`, `ERC721`, `ERC1155` | 标准代币 |
| `Ownable`, `AccessControl` | 权限控制 |
| `Pausable` | 紧急暂停 |
| `ReentrancyGuard` | 防重入攻击 |
| `Upgradeable` | 可升级合约（代理模式） |
| `SafeMath` | 0.8.0 之前防溢出（现在不需要了） |
| `MerkleProof` | Merkle 证明验证 |

**永远先想：OpenZeppelin 有没有这个组件？有就直接用。**

## 6. ERC721：NFT 的标准

ERC20 是同质化代币（每个 1 个都一样）；**ERC721** 是非同质化（每个有唯一 tokenId）。

### 6.1 最简 NFT 合约（用 OpenZeppelin）

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

import "@openzeppelin/contracts/token/ERC721/ERC721.sol";
import "@openzeppelin/contracts/access/Ownable.sol";

contract MyNFT is ERC721, Ownable {
    uint256 private _nextId;
    string  private _baseURI_;

    constructor(string memory baseURI) ERC721("MyNFT", "MNFT") Ownable(msg.sender) {
        _baseURI_ = baseURI;
    }

    function mint(address to) external onlyOwner returns (uint256) {
        uint256 tokenId = _nextId++;
        _safeMint(to, tokenId);
        return tokenId;
    }

    function _baseURI() internal view override returns (string memory) {
        return _baseURI_;
    }
}
```

`_baseURI` + `tokenId` 拼起来就是 NFT 的元数据 URI，指向 IPFS 上的一个 JSON：

```json
{
  "name": "Cool Cat #42",
  "description": "...",
  "image": "ipfs://Qm.../42.png",
  "attributes": [
    { "trait_type": "Color", "value": "Blue" }
  ]
}
```

### 6.2 NFT 项目的真实成本

| 项目 | 成本 |
|---|---|
| 合约部署到主网 | ~$200-500 |
| 单次 mint（用户付） | $5-30 |
| 元数据存储（Arweave/IPFS） | 几十刀 |
| 美术设计 | 视项目大小 |

**"花十万发个 NFT"基本是被中间商坑了**——技术成本不到 1% 的项目

## 7. 6 大安全漏洞（按累计被盗金额排序）

DeFi 史上几乎所有大额被盗事件都因为这几个漏洞之一。**写合约前必背。**

### 7.1 重入攻击（The DAO 2016, 损失 $60M）

**漏洞：** 给用户转 ETH 时，对方合约会执行 fallback 函数，可以再调用你的合约。

```solidity
// ❌ 错误
function withdraw() public {
    uint256 amount = balances[msg.sender];
    (bool ok,) = msg.sender.call{value: amount}("");   // 这里给攻击者合约
    require(ok);
    balances[msg.sender] = 0;       // 太晚了！攻击者已经在 fallback 里再调一次
}

// ✅ 正确：CEI 模式 (Checks-Effects-Interactions)
function withdraw() public {
    uint256 amount = balances[msg.sender];
    balances[msg.sender] = 0;        // 先改状态
    (bool ok,) = msg.sender.call{value: amount}("");
    require(ok);
}

// ✅ 更稳：用 OpenZeppelin 的 ReentrancyGuard
function withdraw() public nonReentrant { ... }
```

### 7.2 整数溢出（Solidity < 0.8.0）

```solidity
uint8 x = 255;
x += 1;             // 0.8.0 之前会变成 0；之后会 revert
```

**修复：** 用 Solidity 0.8.0+，自动检测溢出。老合约用 `SafeMath`。

### 7.3 权限控制错误

```solidity
// ❌ 任何人都能调
function withdrawAll() public {
    payable(msg.sender).transfer(address(this).balance);
}

// ✅
function withdrawAll() public onlyOwner {
    payable(owner()).transfer(address(this).balance);
}
```

历史上**几亿美金**因为忘记 `onlyOwner` 被盗。

### 7.4 价格预言机操纵（Flash Loan 攻击的核心）

```solidity
// ❌ 用一个 DEX 的瞬时价格做抵押估值
uint price = dex.getPrice(token);    // 攻击者可以闪电贷把价格砸下来
require(collateral * price > debt);

// ✅
uint price = chainlink.getPrice(token);   // 用 Chainlink 等抗操纵预言机
// 或者用 TWAP（时间加权平均价）
```

### 7.5 签名重放

```solidity
// ❌ 签名可以被多次使用
function claim(bytes memory sig) public {
    require(verify(msg.sender, sig));
    payable(msg.sender).transfer(1 ether);
}

// ✅ 加 nonce + chainId
function claim(uint256 nonce, bytes memory sig) public {
    require(!used[nonce]);
    require(verify(msg.sender, block.chainid, nonce, sig));
    used[nonce] = true;
    payable(msg.sender).transfer(1 ether);
}
```

### 7.6 tx.origin vs msg.sender

```solidity
// ❌ 用 tx.origin 鉴权
require(tx.origin == owner);    // 攻击者诱骗 owner 调他的合约，他的合约再调你的，绕过

// ✅ 永远用 msg.sender
require(msg.sender == owner);
```

**记住：tx.origin 只在极少数场景用，99% 时候用 msg.sender。**

### 🎯 练习

三个练习：

1. 写一个有漏洞版的银行合约（存钱、取钱、转账），故意留重入漏洞；再写一个攻击者合约把它的钱掏空
2. 把漏洞修好，用 ReentrancyGuard
3. 部署到本地链，用 web3.py 模拟整个攻击流程

In [ ]:
# 在这里写你的答案


## 8. Gas 优化：写合约的钱思维

> **Solidity 不是 Python。每行代码都是真金白银。**

### 8.1 关键 Gas 数字（凭记忆即可）

| 操作 | Gas |
|---|---|
| 加法 / 比较 | 3 |
| 内存读写 | 3 |
| `SSTORE`（写 storage，0→非0） | 20,000 |
| `SSTORE`（写 storage，非0→非0） | 5,000 |
| `SLOAD`（读 storage） | 2,100 |
| 转账（普通） | 21,000 |
| 部署合约（小合约） | 200,000~500,000 |

**结论：** 每次读写 storage 都是大头。优化的核心是**减少 storage 操作**。

### 8.2 五个最有用的 Gas 优化技巧

**1. Storage → Memory 缓存**

```solidity
// ❌ 每次循环都 SLOAD
for (uint i = 0; i < users.length; i++) { ... }    // users.length 是 storage 读

// ✅ 先复制到内存
uint len = users.length;
for (uint i = 0; i < len; i++) { ... }
```

**2. 把多个变量打包到一个 slot**

```solidity
// ❌ 占 3 个 slot
uint256 a;
uint256 b;
uint256 c;

// ✅ 占 1 个 slot（编译器自动打包）
uint128 a;
uint64 b;
uint64 c;
```

**3. 使用 `unchecked` 块**

```solidity
// ❌ Solidity 0.8+ 自动检查溢出
for (uint i = 0; i < n; i++) { ... }

// ✅ 确认不会溢出，跳过检查
for (uint i = 0; i < n;) {
    ...
    unchecked { ++i; }
}
```

**4. 优先 `external` 而不是 `public`**

`external` 函数的参数从 calldata 读取，比 memory 便宜得多。

**5. 用 `events` 替代 storage**

如果只是"记录历史"，不需要链上读，用 event 比存 storage 便宜 100 倍。

## 9. 真实开发流程

### 9.1 工具链

| 工具 | 用途 | 推荐度 |
|---|---|---|
| **Foundry** | 现代 Solidity 工具链（Rust 写的）| ⭐⭐⭐⭐⭐ |
| **Hardhat** | 老牌 JS 工具链 | ⭐⭐⭐⭐ |
| **Remix** | 浏览器 IDE，快速调试用 | ⭐⭐⭐（学习/演示）|
| **Tenderly** | 链上调试、模拟 | ⭐⭐⭐⭐⭐ |

**新项目首选 Foundry**——测试用 Solidity 写，速度比 Hardhat 快 10 倍。

### 9.2 部署流程（生产环境）

```
1. 本地开发（Foundry / Hardhat 本地链）
   ↓
2. 测试网部署（Sepolia / Holesky）
   ↓
3. 单元测试覆盖率 > 90%
   ↓
4. 内部安全审查（自己 + 同事互查）
   ↓
5. 外部审计（CertiK / Trail of Bits / OpenZeppelin Audit）
   ↓ 通常 2-8 周，$50k-500k
6. 测试网公开测试（找用户测）
   ↓
7. 主网部署
   ↓
8. 验证源码（Etherscan）
   ↓
9. 监控 + Bug Bounty（Immunefi）
```

**每跳过一步都可能损失百万美元。**Web3 历史上几乎每个大事故都因为跳过了某一步。

### 9.3 主网部署成本（参考）

| 链 | 简单合约部署 | 一次交易 |
|---|---|---|
| 以太坊主网 | $50-300 | $2-20 |
| Arbitrum | $0.5-3 | $0.01-0.1 |
| Optimism | $0.5-3 | $0.01-0.1 |
| Base | $0.5-3 | $0.01-0.1 |
| Polygon | $0.05-0.5 | $0.001-0.01 |
| BNB Chain | $0.5-3 | $0.05-0.5 |

**学习时优先在 L2 测试网** + **Foundry 本地链**。

## 10. Solidity 学习里程碑

按顺序刷完这些，就达到了**初级 Solidity 工程师**的水平：

```
☐ 用 Remix 部署 HelloWorld，调通 setMessage
☐ 用 OpenZeppelin 模板，部署 ERC20 到 Sepolia
☐ 用 OpenZeppelin 模板，部署 NFT 到 Sepolia，传一个 metadata 到 IPFS
☐ 写一个简单的拍卖合约（出价、结束、退款）
☐ 写一个时间锁定合约（存钱、X 天后才能取）
☐ 写一个多签钱包（N 人中 K 人签名才能转账）
☐ 通过 CryptoZombies 全部章节（免费 + 中文）
☐ 通过 Ethernaut（每关一个安全漏洞，强推）
☐ 通过 Damn Vulnerable DeFi（DeFi 漏洞实战）
☐ 学 Foundry，把你的合约用 forge test 写测试
☐ 阅读一个真实协议代码（如 Uniswap V2 Pair.sol, < 300 行）
```

完成以上，你**已经能投智能合约工程师初级岗位**。

## 11. 本节小结

**你现在应该会的：**

- ✅ 读懂 Solidity 代码
- ✅ 写出 ERC20、NFT 合约
- ✅ 用 web3.py 部署、调用合约、读事件
- ✅ 识别 6 大常见漏洞
- ✅ 知道生产环境用 OpenZeppelin、Foundry、审计

**还差什么（在 06、07 节补）：**

- ⬜ DeFi 协议怎么工作（Uniswap、Aave）
- ⬜ AMM 数学原理
- ⬜ 链上数据分析（Dune、The Graph）
- ⬜ MEV、Flashbots、链上套利

---

**下一节：`06_Web3_DeFi_链上数据.ipynb`**

我们会**手写一个 Uniswap V2**——是的，完整的 AMM 数学 + Python 实现。然后学怎么用 Dune 分析链上数据。